# GridCombat Autoresearch — Colab Edition

**Runtime required:** Gemini 2.5 Flash-Lite API key

| Cell | Type   | Purpose |
|------|--------|---------|
| 1    | Python | Mount Drive, set path constants, export as env vars |
| 2    | Bash   | Install Node.js, zip utilities, and Python packages |
| 3    | Bash   | Project setup: restore from Drive zip or initialise a new directory |
| 4    | Bash   | First run only: copy JS files, make initial save, and save zip to Drive |
| 5    | Python | Gemini 2.5 Flash-Lite API backend |
| 6    | Python | Define all orchestrator functions (read before running 7) |
| 7    | Python | Run the experiment loop |

**Resuming after session expiry:** re-run cells 1, 2, 3, 5, 6, 7. Skip cell 4.

## Cell 1 (Python) — Mount Drive and export paths

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_ROOT  = '/content/drive/MyDrive/gridcombat'
WORK_DIR    = '/content/gridcombat'
MODEL_CACHE = '/content/model_cache'  # local Colab disk (~80 GB free) -- model re-downloads each session
ZIP_PATH    = '/content/drive/MyDrive/gridcombat/repo.zip'

# Export so bash cells can use $DRIVE_ROOT, $WORK_DIR, etc.
os.environ['DRIVE_ROOT']  = DRIVE_ROOT
os.environ['WORK_DIR']    = WORK_DIR
os.environ['MODEL_CACHE'] = MODEL_CACHE
os.environ['ZIP_PATH']    = ZIP_PATH

os.makedirs(DRIVE_ROOT,  exist_ok=True)
os.makedirs(WORK_DIR,    exist_ok=True)
os.makedirs(MODEL_CACHE, exist_ok=True)

print(f'Drive root  : {DRIVE_ROOT}')
print(f'Work dir    : {WORK_DIR}')
print(f'Model cache : {MODEL_CACHE}')
print(f'Zip path    : {ZIP_PATH}')
print('Drive mounted OK.')

## Cell 2 (Bash) — Install Node.js, Zip, and Python packages

In [ ]:
%%bash
echo '--- Node.js & Zip ---'
if ! command -v node &> /dev/null; then
    apt-get install -y nodejs zip unzip 2>&1 | tail -3
fi
node --version

echo '--- Python packages ---'
pip install -q google-generativeai

echo 'Done.'

## Cell 3 (Bash) — Project setup

Restores from Drive zip archive if a previous session exists; otherwise initialises a new directory.
Safe to re-run on session restart.

In [ ]:
%%bash
mkdir -p "$WORK_DIR"
cd "$WORK_DIR"

if [ -f "$ZIP_PATH" ]; then
    echo 'Zip archive found on Drive -- restoring files...'
    unzip -q -o "$ZIP_PATH"
    echo
    echo 'Recent history:'
    tail -n 5 changes.log 2>/dev/null || echo '(no history yet)'
    echo 'Files restored.'
else
    echo 'Fresh directory initialised.'
    echo 'Run Cell 4 to add game files (first run only).'
fi

## Cell 4 (Bash) — Copy game files, initial save, and create zip archive

**First run only. Skip on resume.**

Upload `ai.js`, `baseline_ai.js`, `game_core.js`, `evaluate.js` via the Colab
file browser (left sidebar, upload icon) so they appear at `/content/`. Then run this cell.

**Fix applied:** After copying the JS files this cell now calls `zip`
to persist the files to Drive immediately. Without this step the initial setup would be
lost on session expiry because the working directory `/content/gridcombat` is ephemeral.

In [ ]:
%%bash
UPLOAD_DIR='/content'
cd "$WORK_DIR"

ALL_OK=true
for f in ai.js baseline_ai.js game_core.js evaluate.js; do
    if [ -f "$UPLOAD_DIR/$f" ]; then
        cp "$UPLOAD_DIR/$f" "$WORK_DIR/$f"
        echo "Copied: $f"
    elif [ -f "$WORK_DIR/$f" ]; then
        echo "Already present: $f"
    else
        echo "MISSING: $f -- upload it then re-run this cell"
        ALL_OK=false
    fi
done

if [ "$ALL_OK" = true ]; then
    echo 'initial save' > changes.log
    echo
    echo 'Saving zip archive to Drive...'
    zip -q -j "$ZIP_PATH" ai.js baseline_ai.js game_core.js evaluate.js changes.log && echo "Archive saved to $ZIP_PATH" || echo "ERROR: archive save failed"
    echo
    echo 'Initial save done. Proceed to Cell 5.'
fi

## Cell 5 (Python) — Gemini 2.5 Flash-Lite API backend

**Key setup (one-time):** In the Colab left sidebar click the key icon (Secrets),
add a secret named `GEMINI_API_KEY`, and paste your key as the value.
Enable notebook access for the secret. The key is stored encrypted by Google
and is never written to the notebook file or cell output.

**Resuming:** re-run cells 1, 2, 3, 5, 6, 7. Skip cell 4.

In [ ]:
# Cell 5 — Gemini 2.5 Flash Lite API backend
# Defines call_local() so Cell 6 and Cell 7 require no changes.

import google.generativeai as genai
from google.colab import userdata

# 1. Authenticate FIRST
genai.configure(api_key=userdata.get('GEMINI_API_KEY'))

# 2. Instantiate standard model (Caching not permitted for inputs < 32k tokens)
_gemini_model = genai.GenerativeModel(
    model_name='gemini-2.5-flash-lite',
    system_instruction=(
        'You are an expert game AI engineer. '
        'You reason carefully, make one focused change at a time, '
        'and always follow the output format exactly.'
    )
)

# 3. Define the orchestrator hook
def call_local(prompt):
    response = _gemini_model.generate_content(prompt)
    return response.text

print('Gemini 2.5 Flash Lite ready.')

## Cell 6 (Python) — Define orchestrator

**This cell must be run before Cell 7.** It defines all constants, file helpers,
state utilities, the evaluator, the inference function, and the prompt builder that
Cell 7 depends on. The experiment loop does not start until Cell 7 is run.

Note: Uses pure OS file I/O. Absolutely zero stdout capture, `subprocess`, or pipes.

In [ ]:
import os, re, time, hashlib, shutil
from datetime import datetime

# ── Configuration ─────────────────────────────────────────────────────────────
TEMPERATURE      = 0.3    # changed from 0.4 because of downstream errors in prompt construction
MAX_EXP          = 0      # 0 = run forever; interrupt kernel to stop
EVAL_TIMEOUT_S   = 240
MAX_RETRIES      = 3
MAX_HISTORY_ROWS = 30
MAX_NEW_TOKENS   = 4096

AI_FILE       = 'ai.js'
BASELINE_FILE = 'baseline_ai.js'
RESULTS_FILE  = 'results.tsv'

REQUIRED_STRINGS = ['runAITurn', 'module.exports', 'shouldDefend']


# ── Logging ───────────────────────────────────────────────────────────────────
def log(msg):
    ts = datetime.now().strftime('%H:%M:%S')
    print(f'[{ts}] {msg}', flush=True)


# ── File helpers ──────────────────────────────────────────────────────────────
def read(filename):
    with open(os.path.join(WORK_DIR, filename), 'r', encoding='utf-8') as f:
        return f.read()

def write(filename, content):
    with open(os.path.join(WORK_DIR, filename), 'w', encoding='utf-8') as f:
        f.write(content)

def append_result(commit, win_rate, eval_time, status, description):
    wr   = f'{win_rate:.4f}'  if win_rate  is not None else '0.0000'
    et   = f'{eval_time:.1f}' if eval_time is not None else '0.0'
    desc = description.replace('\t', ' ')[:200]
    with open(os.path.join(WORK_DIR, RESULTS_FILE), 'a', encoding='utf-8') as f:
        f.write(f'{commit}\t{wr}\t{et}\t{status}\t{desc}\n')

def recent_results(n=MAX_HISTORY_ROWS):
    try:
        lines = read(RESULTS_FILE).strip().splitlines()
        header = lines[0] if lines else 'commit\twin_rate\teval_time_s\tstatus\tdescription'
        return '\n'.join([header] + lines[1:][-n:])
    except FileNotFoundError:
        return 'commit\twin_rate\teval_time_s\tstatus\tdescription'

def best_win_rate_from_history():
    best = 50.0
    try:
        for line in read(RESULTS_FILE).strip().splitlines()[1:]:
            parts = line.split('\t')
            if len(parts) >= 4 and parts[3].strip().lower() == 'keep':
                try:
                    wr = float(parts[1])
                    if wr > best: best = wr
                except ValueError:
                    pass
    except FileNotFoundError:
        pass
    return best


# ── State helpers (Pure File I/O for info gathering) ────────────────────────────
def get_file_hash():
    try:
        with open(os.path.join(WORK_DIR, AI_FILE), 'rb') as f:
            return hashlib.md5(f.read()).hexdigest()[:7]
    except Exception:
        return 'unknown'

def backup_state():
    shutil.copy(os.path.join(WORK_DIR, AI_FILE), os.path.join(WORK_DIR, AI_FILE + '.bak'))

def revert_state():
    bak_path = os.path.join(WORK_DIR, AI_FILE + '.bak')
    if os.path.exists(bak_path):
        shutil.copy(bak_path, os.path.join(WORK_DIR, AI_FILE))
    else:
        log('WARNING: Backup file not found. State may be dirty.')

def log_change(message):
    safe = message.replace('"', "'").replace('\n', ' ')[:120]
    with open(os.path.join(WORK_DIR, 'changes.log'), 'a', encoding='utf-8') as f:
        f.write(f'{get_file_hash()} {safe}\n')
    return True

def recent_changes(n=10):
    try:
        lines = read('changes.log').strip().splitlines()
        return '\n'.join(lines[-n:]) or '(no history yet)'
    except Exception:
        return '(no history yet)'

def save_zip():
    files_to_zip = [AI_FILE, BASELINE_FILE, 'game_core.js', 'evaluate.js', RESULTS_FILE, 'changes.log']
    existing = [f for f in files_to_zip if os.path.exists(os.path.join(WORK_DIR, f))]
    files_str = ' '.join(existing)
    zip_path = os.environ.get('ZIP_PATH', '/content/drive/MyDrive/gridcombat/repo.zip')
    rc = os.system(f'zip -q -j "{zip_path}" {files_str}') >> 8
    if rc == 0:
        log(f'  [drive] Archive saved to {zip_path}')
    else:
        log('  [drive] Archive save failed.')


# ── Evaluator (File-Backed Node Wrapper) ──────────────────────────────────────
def run_evaluator():
    # We dynamically create a JS runner that writes directly to disk from inside JS.
    # This entirely avoids Python stdout capturing, pipes, and shell redirection.
    js_wrapper = (
        "const fs = require('fs');\n"
        "const logFile = 'eval_out.log';\n"
        "fs.writeFileSync(logFile, '');\n"
        "const writeLog = (msg) => fs.appendFileSync(logFile, msg);\n"
        "process.stdout.write = writeLog;\n"
        "process.stderr.write = writeLog;\n"
        "console.log = (...args) => writeLog(args.join(' ') + '\\n');\n"
        "console.error = (...args) => writeLog(args.join(' ') + '\\n');\n"
        "const timer = setTimeout(() => {\n"
        f"    writeLog('\\n[eval] TIMEOUT after {EVAL_TIMEOUT_S}s\\n');\n"
        "    process.exit(124);\n"
        f"}}, {EVAL_TIMEOUT_S} * 1000);\n"
        "timer.unref();\n"
        "try {\n"
        "    require('./evaluate.js');\n"
        "} catch(e) {\n"
        "    writeLog('\\n' + (e.stack || String(e)) + '\\n');\n"
        "    process.exit(1);\n"
        "}\n"
    )

    write('runner.js', js_wrapper)

    # Execute without redirection. All output lives securely inside eval_out.log.
    os.system('node runner.js')

    try:
        out = read('eval_out.log')
    except FileNotFoundError:
        out = ''

    wr = re.search(r'^win_rate:\s+([\d.]+)', out, re.MULTILINE)
    et = re.search(r'^eval_time_s:\s+([\d.]+)', out, re.MULTILINE)

    time.sleep(10)

    if not wr:
        log('  [eval] No win_rate in output. Tail:\n' + '\n'.join(out.splitlines()[-20:]))
        return None, None

    return float(wr.group(1)), float(et.group(1)) if et else 0.0


# ── Local inference ───────────────────────────────────────────────────────────

if 'tokenizer' in globals():
    def call_local(prompt):
        messages = [
            {
                'role': 'system',
                'content': (
                    'You are an expert game AI engineer. '
                    'You reason carefully, make one focused change at a time, '
                    'and always follow the output format exactly.'
                )
            },
            {'role': 'user', 'content': prompt},
        ]
        text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([text], return_tensors='pt').to(model.device)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=TEMPERATURE,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        new_ids = output_ids[0][inputs['input_ids'].shape[1]:]
        return tokenizer.decode(new_ids, skip_special_tokens=True)

elif 'call_local' not in globals():
    def call_local(prompt):
        raise RuntimeError("Error: Run Cell 5 (Gemini) or the Appendix (Local GPU) before running this cell.")


# ── Prompt ────────────────────────────────────────────────────────────────────
GAME_CONSTANTS_SUMMARY = """
## Game constants (read-only, from game_core.js)

Unit stats:  type         hp  move  capture  ranged  range
             infantry     10    3     yes      no      -
             mech         12    2     yes      no      -
             tank         10    2     no       no      -
             heavy        16    2     no       no      -
             artillery     8    2     no       yes    3-4
             rocket         7    2     no       yes    3-5

Damage table (attacker rows, defender cols):
             vs:  inf  tank  mech  heavy  arty  rocket
  infantry         5    2     3     2      4     3
  tank             8    6     5     4      5     6
  mech             6    5     5     3      6     5
  heavy           10    8     9     6      7     8
  artillery        9    8     8     6      5     7
  rocket           6   10     9     8      6     5

Terrain defense multiplier (lower = more damage taken):
  plain:0.85  wood:0.70  mountain:0.40  road:1.00  water:impassable

UNIT_VALUE: infantry:10  mech:30  tank:70  heavy:160  artillery:60  rocket:150

Combat: finalDamage = floor(baseDamage * (attacker.hp/maxHp) * terrainDef * (1-homeBonus))
Melee only: defender counter-attacks if alive.  Ranged: no counter.
Capture: capturer must stand on enemy HQ each turn; 2-10 capture points/turn.
Win: capture all enemy HQs, or eliminate all enemy units.

Defend intercept logic (Loop 1 tuning targets in ai.js):
  threshold : 6   -- Manhattan distance from own HQ at which a capturer triggers intercept mode
  weight    : 100 -- Score penalty per tile of distance from the defending unit to the threat
  When threat detected: hold-HQ bonus (+2000) is suppressed; unit pulled toward threat by (distance * weight).
  When no threat in range: hold-HQ bonus active, unit stays on HQ tile.
"""

def build_prompt(ai_js, results_history, experiment_num, best_wr):
    return f"""You are an autonomous AI researcher. Your job is to improve the game AI
in ai.js for a turn-based strategy game by modifying the heuristic decision logic.

## Current experiment: #{experiment_num}
## Best win_rate so far: {best_wr:.4f}  (baseline = 50.0, higher is better)
## Evaluation: 400 games (4 scenarios x 50 x both sides). Noise ~0.7 points.
{GAME_CONSTANTS_SUMMARY}

## Experiment history (results.tsv -- use this to avoid repeating failures):
{results_history}

## Recent changes (last 10) -- do NOT reproduce any of these exactly:
{recent_changes()}

## Current ai.js -- the ONLY file you may modify:
```javascript
{ai_js}
```

## Your task
Make ONE focused change to improve win_rate. Think about what has and has not
worked in the history above. Do not repeat a change that was already discarded.

Good targets:
- Adjusting threat or caution weights at the top of the file
- Adjusting pull weights inside the movement sort logic
- Tweaking defend intercept threshold (currently 6) and weight (currently 100)
- Reordering attack priorities (who fires first)
- Small, scoped heuristic additions (e.g., retreating when hp is low)

## CRITICAL Directives:
1. DO NOT suggest a change if a similar change recently failed in the experiment history.
2. DO NOT call functions that do not already exist in the provided ai.js code.

## Output format -- follow this EXACTLY:
1. Output your change as a strict SEARCH/REPLACE block.
2. The SEARCH block must be an EXACT, character-for-character copy of the lines from ai.js. Indentation must match perfectly.
3. Keep the SEARCH block AS SHORT AS POSSIBLE (2 to 5 lines) to avoid whitespace mismatches, but long enough to be unique.
4. Never use "..." or "// rest of code" inside the blocks.

```javascript
<<<<SEARCH
[exact lines to find]
====
[new replacement lines]
>>>>REPLACE
```

Example:
<<<<SEARCH
const hqPullWeight = 4;
====
const hqPullWeight = 10;
>>>>REPLACE

1. After the block, write exactly one line starting with CHANGE: detailing the reasoning.

Example:
CHANGE: Raised hqPullWeight from 4 to 10 for non-capturers because units were meandering instead of advancing toward the objective.
"""

# ── Response parsing ──────────────────────────────────────────────────────────
def apply_patch(original_code, response_text):
    import re
    # 1. Flexible block extraction (handles 4 or more '=' signs)
    pattern = r'<<<<SEARCH\n(.*?)\n={4,}\n(.*?)\n>>>>REPLACE'
    match = re.search(pattern, response_text, re.DOTALL)

    if not match:
        return None, "Format error: Use <<<<SEARCH, ====, and >>>>REPLACE exactly."

    search_text = match.group(1).strip()
    replace_text = match.group(2).strip()

    if not search_text:
        return None, "SEARCH block is empty."

    # 2. Avoid Regex for matching to prevent "Hangs"
    # We try an exact match first
    if search_text in original_code:
        new_code = original_code.replace(search_text, replace_text, 1)
        return new_code, "ok"

    # 3. Fallback: Line-by-line normalization (handles indentation mismatches)
    orig_lines = original_code.splitlines()
    search_lines = search_text.splitlines()

    for i in range(len(orig_lines) - len(search_lines) + 1):
        potential_match = True
        for j in range(len(search_lines)):
            if orig_lines[i+j].strip() != search_lines[j].strip():
                potential_match = False
                break

        if potential_match:
            # Reconstruct the code
            final_lines = orig_lines[:i] + [match.group(2)] + orig_lines[i+len(search_lines):]
            return "\n".join(final_lines), "ok"

    return None, "SEARCH block not found. Ensure you copy the lines EXACTLY as they appear."

def extract_description(text):
    after_code = re.sub(r'```.*?```', '', text, flags=re.DOTALL).strip()
    m = re.search(r'^CHANGE:\s*(.+)', after_code, re.MULTILINE | re.IGNORECASE)
    if m: return m.group(1).strip()[:200]
    for line in after_code.splitlines():
        line = line.strip()
        if line and not line.startswith('`'):
            return line[:200]
    return 'no description provided'

def passes_sanity_check(code):
    for s in REQUIRED_STRINGS:
        if s not in code:
            return False, f'missing required string: {s}'
    if len(code) < 500:    return False, f'too short ({len(code)} chars)'
    if len(code) > 80_000: return False, f'too long ({len(code)} chars)'
    return True, 'ok'

def check_js_syntax(code):
    write('_tmp_check.js', code)
    rc = os.system('node --check _tmp_check.js 2> _syn_err.txt') >> 8
    if rc != 0:
        err = read('_syn_err.txt').strip()[:400]
        line_match = re.search(r':(\d+)$', err, re.MULTILINE)
        line_num = line_match.group(1) if line_match else 'unknown'
        log(f'Syntax error at line {line_num}:\n{err}')
        return False, err
    return True, ''

print('Orchestrator defined. Run Cell 7 to start.')

## Cell 7 (Python) — Run the experiment loop

Stop at any time with **Runtime > Interrupt execution**.
Archive is saved to Drive on every KEEP so progress survives session expiry.
On session restart re-run cells 1, 2, 3, 5, 6, then this cell.

**Note:** To inject a manual algorithmic change (Loop 2), interrupt this cell, follow the injection instructions in the architecture section below, and then resume.

In [ ]:
import os
os.chdir(WORK_DIR)

for f in [AI_FILE, BASELINE_FILE]:
    if not os.path.exists(os.path.join(WORK_DIR, f)):
        raise FileNotFoundError(f'{f} not found in {WORK_DIR}. Run Cell 4 first.')

if not os.path.exists(os.path.join(WORK_DIR, RESULTS_FILE)):
    write(RESULTS_FILE, 'commit\twin_rate\teval_time_s\tstatus\tdescription\n')

if 'tokenizer' in globals():
    log('Model           : Qwen2.5 Local (4-bit NF4)')
elif 'genai' in globals() or '_gemini_model' in globals():
    log('Model           : Gemini 2.5 Flash Lite (API)')
else:
    log('Model           : Unknown / Not Loaded')
log(f'Temperature     : {TEMPERATURE}')
log(f'Max experiments : {"inf" if MAX_EXP == 0 else MAX_EXP}')
log(f'Eval timeout    : {EVAL_TIMEOUT_S}s')
log(f'Work dir        : {WORK_DIR}')
log(f'Drive archive   : {os.environ.get("ZIP_PATH", "")}')
log('')

best_wr              = best_win_rate_from_history()
experiment_num       = 0
consecutive_failures = 0
log(f'Best win_rate from history: {best_wr:.4f}')

while True:
    experiment_num += 1
    if MAX_EXP > 0 and experiment_num > MAX_EXP:
        log(f'Reached MAX_EXPERIMENTS={MAX_EXP}. Stopping.')
        break

    log('')
    log('=' * 60)
    log(f'Experiment #{experiment_num}  |  Best: {best_wr:.4f}')
    log('=' * 60)

    # ---- Inference ----
    ai_js  = read(AI_FILE)
    prompt = build_prompt(ai_js, recent_results(), experiment_num, best_wr)
    log('Running local inference...')
    try:
        response_text        = call_local(prompt)
        consecutive_failures = 0
    except Exception as e:
        error_str = str(e).lower()
        if '429' in error_str or 'exhausted' in error_str or 'quota' in error_str:
            consecutive_failures += 1
            wait_time = 60 * consecutive_failures  # 60s, then 120s, then 180s...
            log(f'QUOTA EXHAUSTED. Sleeping for {wait_time} seconds...')
            import time
            time.sleep(wait_time)
            continue

        log(f'Inference error: {e}')
        consecutive_failures += 1
        if consecutive_failures >= MAX_RETRIES:
            log('Too many consecutive failures. Stopping.')
            break
        experiment_num -= 1
        import time; time.sleep(5)
        continue

    # ---- Parse ----
    new_code, err_msg = apply_patch(ai_js, response_text)
    if new_code:
        print(re.search(r'<<<<SEARCH.*?>>>>REPLACE', response_text, re.DOTALL).group(0))
    description = extract_description(response_text) or "Inference change"

    if new_code is None:
        log(f'Patch failed: {err_msg}. Skipping.')
        log('Preview: ' + response_text[:400].replace('\n', ' '))
        experiment_num -= 1
        continue

    ok, reason = passes_sanity_check(new_code)
    if not ok:
        log(f'Sanity check failed: {reason}. Skipping.')
        experiment_num -= 1
        continue

    syn_ok, syn_err = check_js_syntax(new_code)
    if not syn_ok:
       log(f'Syntax error (node --check):\n{syn_err}')
       experiment_num -= 1
       continue

    log(f'Proposed: {description}')

    # ---- Modify & Backup ----
    backup_state()
    write(AI_FILE, new_code)
    log_change(description)

    commit_hash = get_file_hash()
    log(f'Testing {commit_hash}')

    # ---- Evaluate ----
    log('Running evaluator...')
    win_rate, eval_time = run_evaluator()

    if win_rate is None:
        log('CRASH -- evaluator returned no win_rate. Reverting.')
        revert_state()
        append_result(commit_hash, None, None, 'crash', description)
        continue

    delta = win_rate - best_wr
    log(f'win_rate: {win_rate:.4f}  ({delta:+.4f} vs best)  eval_time: {eval_time:.1f}s')

    # ---- Keep or discard ----
    if win_rate > best_wr:
        best_wr = win_rate
        log(f'KEEP -- new best: {best_wr:.4f}')
        append_result(commit_hash, win_rate, eval_time, 'keep', description)
        save_zip()
    else:
        revert_state()
        log('DISCARD -- reverted to previous best.')
        append_result(commit_hash, win_rate, eval_time, 'discard', description)

## Architecture Note — Two-Loop AI Development Strategy

This notebook implements one of two intended research loops. They operate at different levels and reinforce each other.

### Loop 1 — Autoresearch (This Notebook)
Operates continuously. The model (Gemini 2.5 Flash-Lite) makes small, focused changes to `ai.js` guided solely by win rate. It excels at parametric optimisation (weights, thresholds) and discovering emergent heuristic improvements, but cannot reliably produce large architectural changes.

### Loop 2 — Directed Research (Human-Guided)
Operates through human observation. Known gameplay problems are reasoned about to produce targeted algorithmic fixes addressing specific diagnosed failures.

### Division of Labour
| Class of change | Loop 1 (Auto) | Loop 2 (Manual) |
|---|---|---|
| Parameter tuning & Simple heuristics | Yes | No |
| Emergent behavioural correction | Yes | No |
| Targeted fix for a known problem / Substantial algorithms | No | Yes |

### Injecting a Loop 2 Change
1. Interrupt Cell 7 if running.
2. Edit `ai.js` in `/content/gridcombat/` directly with the targeted change.
3. Log it manually: `echo 'directed: description' >> changes.log`
4. Save the archive: `zip -q -j "$ZIP_PATH" ai.js baseline_ai.js game_core.js evaluate.js results.tsv changes.log`
5. Resume Cell 7. The autoresearch loop will continue from the new baseline.

## Re-Rooting Protocol — LLM-Based Monte Carlo Tree Search

### The MCTS Analogy

Loop 1 is structurally equivalent to Monte Carlo Tree Search (MCTS). Each experiment is a simulation: the LLM proposes a move (a code change), the evaluator runs a playout (400 games), and the keep/revert decision propagates the result back. The experiment history fed into each prompt is the tree memory — it biases future proposals away from exhausted branches and toward unexplored ones.

As in MCTS, the search tree has a root node. The root is defined by the current `baseline_ai.js`. All win rates are measured relative to this root. When Loop 1 finds a new best, it has discovered a productive child node. When it plateaus — repeatedly returning the same win rate across many experiments — it has exhausted the subtree reachable from the current root.

### The Plateau Signal

A plateau of 6 or more consecutive experiments at the same win rate is the operational signal that the current subtree is exhausted. This has two causes:

1. **Conditioning on stale history.** The experiment history accumulates failed attempts calibrated against the old baseline. The model avoids changes that resemble past failures, even if those changes would be beneficial at the new win rate level. The history has become a constraint rather than a guide.
2. **Signal resolution.** Small genuine improvements above the current best are indistinguishable from noise (~0.7 points) when the model is proposing changes informed by a search space mapped at a lower win rate.

### The Re-Rooting Operation

Re-rooting is the canonical MCTS operation for an exhausted subtree: commit to the best child node and make it the new root. In this pipeline that means:

**Before re-rooting — Loop 2 validation (mandatory):**
1. Human review of all committed diffs in `changes.log` — confirm each kept change is tactically coherent, not a numerical accident exploiting an evaluator quirk.
2. Manual play-test of the committed `ai.js` — confirm AI behaviour is qualitatively better, not just statistically better.
3. Sanity check for map overfitting — 400 games across 4 maps is robust, but a spot check on a map outside the evaluation set is advisable.

**The re-rooting operation itself:**
1. Interrupt Cell 7 if running.
2. `cp ai.js baseline_ai.js` — promote the validated best to the new root.
3. `echo 'commit\twin_rate\teval_time_s\tstatus\tdescription' > results.tsv && echo 'rerooted\t50.0000\t0\tRE-ROOT\tBaseline promoted from prior session best' >> results.tsv` — reset history with a clean root marker.
4. Update `best_wr = 50.0` in the session state.
5. Save the archive: `zip -q -j "$ZIP_PATH" ai.js baseline_ai.js game_core.js evaluate.js results.tsv changes.log`
6. Resume Cell 7. Loop 1 now searches from the new node with a clean history.

### Why the Win Rate Clock Resets to 50%

Resetting to 50% is not losing progress — it is re-rooting the search. The new `baseline_ai.js` embodies all prior gains. The evaluator measures `ai.js` against `baseline_ai.js`, so 50% now means parity with a stronger opponent than before. Any improvement Loop 1 finds above 50% in the new session is additive depth on top of all prior sessions.

This makes the pipeline an **anytime algorithm**: at any point the current `ai.js` is the best known solution; Loop 2 validates and re-roots; Loop 1 continues searching from the new node indefinitely. Each session adds one layer of depth to the search tree. The win rate ceiling is not fixed — it rises with each re-rooting cycle.

### Higher-Dimensional Navigation

The win rate is a low-dimensional, uninterpretable signal — it cannot distinguish between mechanisms. Loop 2 operates in a higher-dimensional space: it reasons from game mechanics, unit statistics, combat formulas, and movement constraints to derive targeted fixes that the evaluator cannot discover by search alone. These fixes are projected down into the scoring space as new parameters with zero defaults, preserving the baseline exactly until Loop 1 finds nonzero values worth keeping.

The two loops are therefore not in competition. Loop 2 navigates the high-dimensional space to find structurally sound branches. Loop 1 exhausts those branches parametrically. Re-rooting connects the two: it promotes a Loop 1 result into Loop 2 territory for validation, then hands the validated node back to Loop 1 as a new root. The cycle repeats indefinitely.

### Summary Table

| Signal | Meaning | Action |
|---|---|---|
| Win rate rising | Productive subtree | Keep running Loop 1 |
| Win rate plateau (6+ experiments) | Subtree exhausted | Trigger Loop 2 validation and re-root |
| Win rate regression then recovery | Correct revert behaviour | No action needed |
| Loop 2 structural fix available | New branch to explore | Inject via Loop 2 protocol, then re-root |


## Findings, Conclusions, and Future Directions

### What Was Established

This system constitutes a formal instrument for autonomous game AI research. The following was established empirically:

The autoresearch loop functions correctly and reliably when using Gemini 2.5 Flash-Lite. It handles inference, evaluation, and persistence without human intervention. In a single session, win rate advanced from 50.0% to 62.5% across three committed experiments — a gain of 12.5 points well above the evaluator noise floor of ~0.7 points. This confirms the pipeline produces real, statistically significant results and not numerical artefacts.

The session also demonstrated correct revert behavior: One experiment regressed to 50.0% and was correctly discarded, with the pipeline recovering to 62.5% on the next attempt without human intervention.

Earlier experiments with Qwen 2.5 3B demonstrated that while 3B-class models can achieve initial gains, they eventually hit a capacity ceiling in agentic use. The transition to Gemini 2.5 Flash-Lite successfully bypassed these limitations, allowing for sustained, autonomous exploration of the heuristic search space.

### The Plateau and the MCTS Discovery

Following the 62.5% peak, the session entered a plateau of 8 consecutive experiments at the same win rate. This plateau was identified as the operational signal of an exhausted subtree in the LLM-based MCTS framework — see the Re-Rooting Protocol section above. The plateau is not a failure; it is the expected and correct terminal condition for a Loop 1 session, indicating that the current search space has been fully explored and re-rooting is required to continue making progress.

### The Boundary Condition

The autoresearch loop is a tool for capturing improvements through parameter and heuristic search. It is highly effective at optimising existing logic. Discovering entirely novel algorithms remains a task for the human-guided loop, which provides the structural baseline for the autonomous loop to refine.

### Future Directions

1. **Re-rooting cycle.** Validate the current 62.5% `ai.js` via Loop 2 human review and play-testing, then promote to `baseline_ai.js` and begin a new Loop 1 session from the re-rooted 50% baseline.
2. **Spatial screening parameter.** A mechanically proven ranged unit screening parameter (`SPATIAL_SUPPORT_WEIGHT`) was derived from combat formulas and tested empirically across multiple sessions using both Gemini 2.5 Flash-Lite and Qwen3 235B. The evaluator returned no measurable win rate improvement. Conclusion: the four test maps do not generate the exposed artillery scenario frequently enough for the parameter to have a statistically detectable effect. The existing cohesion bonus and general unit clustering provide sufficient implicit protection. Patch discarded.
3. **Human play testing.** The current best AI must be tested against human opponents to verify that win rate gains translate to increased tactical difficulty.
4. **Richer context.** Feeding more detailed game logs into the prompt to provide the model with a better understanding of why certain games were lost, potentially narrowing the search toward even more productive changes.

### A Note on the Contribution

This notebook documents the reasoning, constraints, and successes of the autoresearch approach. It provides a reproducible foundation for iterative AI improvement, demonstrating how API-based models can be integrated into an autonomous research harness to overcome local hardware limitations. The identification of the pipeline as an LLM-based MCTS anytime algorithm — with re-rooting as the mechanism for unbounded depth — is the principal theoretical contribution of this session.

## Appendix — Local Model Execution (Qwen 2.5 3B)

**Historical Documentation / Alternative Method**

This code was previously used as the primary inference engine before transitioning to the Gemini API. It allows the notebook to run entirely locally on a free Colab T4 GPU without any external API calls.

To use this instead of Gemini (Cell 5):
1. Change the Colab runtime to **T4 GPU**.
2. Add `!pip install -q transformers accelerate bitsandbytes` to Cell 2 and run it.
3. Run this cell *instead* of Cell 5.
4. Proceed with Cells 6 and 7 as normal. The orchestrator is designed to automatically hook into this local model if it detects it.

`Qwen/Qwen2.5-3B-Instruct` is a public, ungated model — no HuggingFace account or token is required.

Downloads approximately 6 GB on first run to local Colab disk (`/content/model_cache`). Download time is approximately 3-4 minutes on a new session. The model is not cached to Drive and will re-download at the start of each session. 4-bit NF4 quantization uses approximately 4-5 GB VRAM on a T4 (16 GB total), leaving sufficient headroom for inference. The 7B model was found to consume 13.62 GB of the 14.56 GB available, leaving insufficient memory for the inference buffer. The 3B model is used instead. `PYTORCH_ALLOC_CONF=expandable_segments:True` is set to reduce memory fragmentation and `MAX_NEW_TOKENS` is set to 4096, which provides sufficient generation length.

In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f'Loading tokenizer: {MODEL_ID}')
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, cache_dir=MODEL_CACHE, trust_remote_code=True
)

print('Loading model at 4-bit NF4...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    cache_dir=MODEL_CACHE,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.float16,
    attn_implementation="sdpa"
)
model.eval()

print(f'Device : {next(model.parameters()).device}')
if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM   : {used:.1f} GB used / {total:.1f} GB total')
print('Model ready.')